# Compact three-state histogram figure

This notebook reruns the three Monte Carlo simulations and plots the T=50 grouped histogram plus one whole-population KDE.

In [ ]:
"""Compact, reproducible version of the three-state MC figure."""

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde


# Locate the repository whether this file is run from the root or notebooks/.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root.parent))

from audit_panel_alignment import require_matplotlib_panel_alignment
from src.initial_condition_states import simulate_asymmetric_bimodal
from src.monte_carlo import MonteCarloParameters, simulate_trajectory


# ---------------------------- model and simulations ----------------------------

common = dict(
    n=800,
    k=40,
    t_max=50,
    f_s=0.50,
    sigma_s=0.80,
    sigma_ns=0.10,
    gamma=4.0,
    d_c=0.50,
    trajectory_seed=123,
    trajectories_per_bias_group=200,
)

parameters = {
    "bp": MonteCarloParameters(
        **common,
        seed=80,
        x0_nonstubborn_negative=-0.5,
        x0_nonstubborn_positive=0.5,
    ),
    "bu": MonteCarloParameters(
        **common,
        seed=80,
        x0_nonstubborn_negative=0.5,
        x0_nonstubborn_positive=0.5,
    ),
    "ab": MonteCarloParameters(
        **common,
        seed=62,
        x0_nonstubborn_negative=-0.5,
        x0_nonstubborn_positive=0.5,
    ),
}

# Every run generates fresh MC trajectories. No CSV or old figure is loaded.
results = {
    "bp": simulate_trajectory(0.03, parameters["bp"]),
    "bu": simulate_trajectory(0.03, parameters["bu"]),
    "ab": simulate_asymmetric_bimodal(0.03, parameters["ab"]),
}

cases = [
    dict(
        panel="(a)",
        key="bp",
        title="Bimodal polarized",
        mf=(-0.7518939448845263, 0.7518939448845263),
        density_color="#6F4E9C",
    ),
    dict(
        panel="(b)",
        key="bu",
        title="Biased unimodal",
        mf=(0.5514161221836354, 0.6679211791156909),
        density_color="#D9772B",
    ),
    dict(
        panel="(c)",
        key="ab",
        title="Asymmetric bimodal polarized",
        mf=(results["ab"]["negative_mf"], results["ab"]["positive_mf"]),
        density_color="#6F4E9C",
    ),
]

# Check the three initial conditions used in the reference figure.
expected_initial_means = {"bp": (-0.5, 0.5), "bu": (0.5, 0.5), "ab": (0.0, 0.5)}
for key, expected in expected_initial_means.items():
    actual = (results[key]["negative_mean"][0], results[key]["positive_mean"][0])
    assert np.allclose(actual, expected)


# ----------------------------------- plotting ----------------------------------

BLUE, RED = "#2166AC", "#B2182B"
LIGHT_BLUE, LIGHT_RED = "#92C5DE", "#F4A6B6"
MF_BLUE, MF_RED = "#00A6D6", "#EF3B6B"
KDE_BANDWIDTH = 0.075

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 8.0,
        "axes.labelsize": 12.0,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "axes.linewidth": 0.75,
        "pdf.fonttype": 42,
        "svg.fonttype": "none",
    }
)


def style_axis(ax):
    ax.tick_params(direction="out", top=False, right=False, width=0.7, length=3)
    for spine in ax.spines.values():
        spine.set_linewidth(0.75)


def draw_trajectory(ax, result, mf):
    negative = result["sampled_is_negative"]
    history = result["sampled_history"]
    times = result["times"]
    ax.plot(times, history[:, negative], color=LIGHT_BLUE, alpha=0.10, lw=0.35)
    ax.plot(times, history[:, ~negative], color=LIGHT_RED, alpha=0.10, lw=0.35)
    ax.plot(times, result["negative_mean"], color=BLUE, lw=1.25)
    ax.plot(times, result["positive_mean"], color=RED, lw=1.25)
    ax.axhline(mf[0], color=MF_BLUE, ls="--", lw=1.05)
    ax.axhline(mf[1], color=MF_RED, ls="--", lw=1.05)
    ax.set(xlim=(0, 50), ylim=(-1.02, 1.02), xlabel=r"$t$", ylabel=r"$x_i(t)$")
    ax.set_xticks([0, 10, 20, 30, 40, 50])
    ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    style_axis(ax)


def draw_distribution(ax, result, mf, density_color):
    bins = np.linspace(-1, 1, 31)
    bin_width = bins[1] - bins[0]
    groups = [result["final_negative_opinions"], result["final_positive_opinions"]]
    total = sum(len(group) for group in groups)

    # These are the T=50 opinions produced by the simulations above.
    # The weights make the total histogram area equal to one.
    weights = [np.full(len(group), 1 / (total * bin_width)) for group in groups]
    ax.hist(
        groups,
        bins=bins,
        weights=weights,
        stacked=True,
        color=[BLUE, RED],
        alpha=0.32,
        edgecolor="none",
    )

    # One KDE for the complete non-stubborn population, not one KDE per group.
    values = result["final_opinions"]
    grid = np.linspace(-1, 1, 1000)
    kde = gaussian_kde(values, bw_method=KDE_BANDWIDTH / np.std(values, ddof=1))
    density = kde(grid)
    ax.fill_between(grid, 0, density, color=density_color, alpha=0.10)
    ax.plot(grid, density, color=density_color, lw=1.35)

    ax.axvline(result["negative_mean"][-1], color=BLUE, lw=1.25)
    ax.axvline(result["positive_mean"][-1], color=RED, lw=1.25)
    ax.axvline(mf[0], color=MF_BLUE, ls="--", lw=1.05)
    ax.axvline(mf[1], color=MF_RED, ls="--", lw=1.05)
    ax.set(xlim=(-1, 1), ylim=(0, None), xlabel=r"$x_i$", ylabel=r"$p(x_i)$")
    ax.set_xticks([-1, -0.5, 0, 0.5, 1])
    style_axis(ax)


fig, axes = plt.subplots(
    2,
    3,
    figsize=(7.20, 4.70),
    dpi=300,
    gridspec_kw={"height_ratios": [1.0, 0.78]},
)
fig.subplots_adjust(left=0.075, right=0.992, bottom=0.115, top=0.715, wspace=0.34, hspace=0.56)

for column, case in enumerate(cases):
    result = results[case["key"]]
    draw_trajectory(axes[0, column], result, case["mf"])
    draw_distribution(axes[1, column], result, case["mf"], case["density_color"])
    axes[0, column].text(
        0.5,
        1.10,
        case["title"],
        transform=axes[0, column].transAxes,
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )
    axes[0, column].text(
        -0.14,
        1.10,
        case["panel"],
        transform=axes[0, column].transAxes,
        fontsize=10.5,
        fontweight="bold",
    )
    axes[1, column].text(
        0.5,
        1.015,
        rf"$B={result['B']:.3f},\quad S={result['S']:.3f}$",
        transform=axes[1, column].transAxes,
        ha="center",
        va="bottom",
        fontsize=8,
    )

legend_handles = [
    Line2D([0], [0], color=BLUE, lw=1.25, label="Negative-bias MC mean"),
    Line2D([0], [0], color=RED, lw=1.25, label="Positive-bias MC mean"),
    Line2D([0], [0], color=MF_BLUE, ls="--", lw=1.05, label="Negative-bias MF"),
    Line2D([0], [0], color=MF_RED, ls="--", lw=1.05, label="Positive-bias MF"),
]
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.82),
    ncol=4,
    frameon=True,
    fontsize=6.2,
    handlelength=2.2,
    columnspacing=0.9,
)

output = repo_root / "figures" / "three_states_K40_T50_histogram_compact"
require_matplotlib_panel_alignment(
    fig,
    json_out=str(output) + ".alignment.json",
    overlay_svg=str(output) + ".alignment.svg",
    tolerance_pt=1.5,
    gutter_tolerance_pt=1.5,
    strict=True,
)
fig.savefig(str(output) + ".png", dpi=600, bbox_inches="tight")
fig.savefig(str(output) + ".pdf", bbox_inches="tight")
fig.savefig(str(output) + ".svg", bbox_inches="tight")
plt.show()
